# Link-Kind Audit

Classifies every NCIt and LOINC link in the `cdisc-for-ai` reference files by
**link kind** and **provenance class**, and counts the actual rows behind each.

Purpose: decide one predicate per link kind before rendering DSS/BC content as
RDF in `cosmos-rdf` and `usdm-rdf`. A single `skos:exactMatch` currently hides
at least five different statements.

**Link kinds**

| Kind | Statement |
|---|---|
| `identity` | this thing *is* that NCIt concept (the C-code is its identifier) |
| `membership` | this concept is a permissible value in that codelist |
| `binding` | this variable draws its values from that codelist |
| `denotation` | this model element *means* that concept |
| `pinned_value` | this variable is fixed to that value in this specialization |
| `value_list` | this variable is restricted to these values in this specialization |
| `category_tag` | this BC carries that category token - the object end is a label, not an identifier |
| `mapping` | this concept corresponds to a concept in another terminology |

**Provenance classes**

| Class | Meaning |
|---|---|
| `published` | CDISC / NCI EVS asserts it in a release |
| `derived` | computed by a rule from published inputs |
| `curated` | a judgement, needs a justification |

Counts are measured from the workbooks. The kind/class assignment in
`INVENTORY` is authored in this notebook and is the part to review.

In [ ]:
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent.parent  # notebooks/ -> link-semantics/ -> repo root
OUT = REPO / "link-semantics" / "interim" / "Link_Kind_Audit.xlsx"

GRAPH = REPO / "cosmos-graph" / "interim" / "COSMoS_Graph.xlsx"
GRAPH_CT = REPO / "cosmos-graph" / "interim" / "COSMoS_Graph_CT.xlsx"
TEST_IDENTITY = REPO / "sdtm-test-codes" / "machine_actionable" / "SDTM_Test_Identity.xlsx"
INSTRUMENT_IDENTITY = REPO / "sdtm-test-codes" / "machine_actionable" / "SDTM_Instrument_Identity.xlsx"
CROSS_REFS = REPO / "sdtm-test-codes" / "interim" / "Codelist_Cross_References.xlsx"

for path in [GRAPH, GRAPH_CT, TEST_IDENTITY, INSTRUMENT_IDENTITY, CROSS_REFS]:
    if not path.exists():
        raise FileNotFoundError(f"{path} not found - run this notebook from link-semantics/notebooks/")

print("repo:", REPO)

In [ ]:
SHEETS = {}


def sheet(path, name):
    key = (path.name, name)
    if key not in SHEETS:
        SHEETS[key] = pd.read_excel(path, sheet_name=name)
    return SHEETS[key]

## Pins

What this run was executed against. The SDTM CT release date is not recorded
in any of the input files (the extract reads a versionless NCI EVS URL), so it
cannot be pinned from data here.

In [ ]:
import subprocess


def generated(path):
    book = pd.ExcelFile(path)
    readme_name = [name for name in book.sheet_names if name.lower() == "readme"][0]
    readme = pd.read_excel(book, sheet_name=readme_name, header=None)[0].dropna().astype(str)
    return readme[readme.str.startswith("Generated")].iloc[0].split(":", 1)[1].strip()


commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True, check=True
).stdout.strip()
package_dates = sheet(GRAPH, "DSS")["package_date"]

pins = pd.DataFrame(
    [{"pin": "cdisc-for-ai commit", "value": commit}]
    + [{"pin": "COSMoS package_date (latest in DSS sheet)", "value": str(package_dates.max())}]
    + [{"pin": "COSMoS package_date (distinct values)", "value": str(package_dates.nunique())}]
    + [
        {"pin": f"{path.name} generated", "value": generated(path)}
        for path in [GRAPH, GRAPH_CT, TEST_IDENTITY, INSTRUMENT_IDENTITY, CROSS_REFS]
    ]
)
pins

## Inventory

One row per place where a link is stored. `subject` and `object` name the
columns that carry the two ends. `filter` is a pandas query applied first.
`justification` names a column that already records *how* the link was
established, or `None` where none exists.

In [ ]:
INVENTORY = [
    {
        "link_kind": "identity",
        "statement": "BC is an NCIt concept",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "BC",
        "subject": ["bc_id"],
        "object": "ncit_code",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "identity",
        "statement": "DEC is an NCIt concept",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "DataElementConcepts",
        "subject": ["dec_id"],
        "object": "ncit_dec_code",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "identity",
        "statement": "TESTCD is an NCIt concept",
        "provenance": "published",
        "path": TEST_IDENTITY,
        "sheet": "Test Codes",
        "subject": ["TESTCD"],
        "object": "NCIt_Code",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "identity",
        "statement": "codelist is an NCIt concept",
        "provenance": "published",
        "path": GRAPH_CT,
        "sheet": "Codelists",
        "subject": ["codelist_submission_value"],
        "object": "codelist_concept_id",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "membership",
        "statement": "concept is a permissible value in codelist",
        "provenance": "published",
        "path": GRAPH_CT,
        "sheet": "CodelistTerms",
        "subject": ["codelist_concept_id"],
        "object": "term_concept_id",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "binding",
        "statement": "variable draws values from codelist",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "codelist_concept_id",
        "filter": "codelist_concept_id.notna()",
        "justification": None,
    },
    {
        "link_kind": "denotation",
        "statement": "variable means the DEC",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "dec_id",
        "filter": "dec_id.notna()",
        "justification": None,
    },
    {
        "link_kind": "pinned_value",
        "statement": "variable is fixed to an NCIt-coded value in this DSS",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "assigned_term_concept_id",
        "filter": "assigned_term_concept_id.notna()",
        "justification": None,
    },
    {
        "link_kind": "pinned_value",
        "statement": "DSS pins an external LOINC code on a --LOINC variable",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "assigned_term_value",
        "filter": "variable_name.str.endswith('LOINC') and assigned_term_value.notna()",
        "justification": None,
    },
    {
        "link_kind": "value_list",
        "statement": "variable is restricted to a list of values from its bound codelist in this DSS",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "value_list",
        "filter": "value_list.notna() and codelist_concept_id.notna()",
        "justification": None,
    },
    {
        "link_kind": "value_list",
        "statement": "variable is restricted to a list of values with no bound codelist in this DSS",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "value_list",
        "filter": "value_list.notna() and codelist_concept_id.isna() and not variable_name.str.endswith('LOINC')",
        "justification": None,
    },
    {
        "link_kind": "value_list",
        "statement": "DSS lists several external LOINC codes on a --LOINC variable",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Variables",
        "subject": ["ds_id", "variable_name"],
        "object": "value_list",
        "filter": "variable_name.str.endswith('LOINC') and value_list.notna()",
        "justification": None,
    },
    {
        "link_kind": "category_tag",
        "statement": "BC carries a category token (label, no identifier)",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "BC_Categories",
        "subject": ["bc_id"],
        "object": "category",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "mapping",
        "statement": "BC corresponds to an external code (Coding sheet)",
        "provenance": "published",
        "path": GRAPH,
        "sheet": "Coding",
        "subject": ["bc_id"],
        "object": "code",
        "filter": None,
        "justification": None,
    },
    {
        "link_kind": "mapping",
        "statement": "instrument codelist corresponds to an NCIt instrument concept (C20993)",
        "provenance": "curated",
        "path": INSTRUMENT_IDENTITY,
        "sheet": "Instruments",
        "subject": ["Codelist_TESTCD"],
        "object": "Instrument_NCIt_Code",
        "filter": "Instrument_NCIt_Code.notna()",
        "justification": "Instrument_Match_Method",
    },
    {
        "link_kind": "mapping",
        "statement": "instrument codelist corresponds to an NCIt question container (C211913)",
        "provenance": "derived",
        "path": INSTRUMENT_IDENTITY,
        "sheet": "Instruments",
        "subject": ["Codelist_TESTCD"],
        "object": "Container_NCIt_Code",
        "filter": "Container_NCIt_Code.notna()",
        "justification": None,
    },
    {
        "link_kind": "mapping",
        "statement": "codelist pair declared semantically related",
        "provenance": "curated",
        "path": CROSS_REFS,
        "sheet": "Cross_Reference_Pairs",
        "subject": ["codelist_A_concept_id"],
        "object": "codelist_B_concept_id",
        "filter": None,
        "justification": "rationale",
    },
    {
        "link_kind": "membership",
        "statement": "term overlap/gap between a declared codelist pair",
        "provenance": "derived",
        "path": CROSS_REFS,
        "sheet": "Term_Diff",
        "subject": ["pair_id", "term_concept_id"],
        "object": "gap_classification",
        "filter": None,
        "justification": None,
    },
]

print(len(INVENTORY), "inventory entries")

In [ ]:
def measure(entry):
    df = sheet(entry["path"], entry["sheet"])
    if entry["filter"] is not None:
        df = df.query(entry["filter"], engine="python")
    subject_cols = entry["subject"]
    object_col = entry["object"]
    rows = len(df)
    distinct_subjects = len(df[subject_cols].drop_duplicates())
    distinct_objects = df[object_col].nunique(dropna=True)
    missing_object = int(df[object_col].isna().sum())
    return {
        "link_kind": entry["link_kind"],
        "provenance": entry["provenance"],
        "statement": entry["statement"],
        "file": entry["path"].name,
        "sheet": entry["sheet"],
        "subject_cols": " + ".join(subject_cols),
        "object_col": object_col,
        "rows": rows,
        "distinct_subjects": distinct_subjects,
        "distinct_objects": distinct_objects,
        "object_missing": missing_object,
        "justification_col": entry["justification"] or "",
    }


audit = pd.DataFrame([measure(entry) for entry in INVENTORY])
audit

## Summary by link kind and provenance

In [ ]:
summary = (
    audit.groupby(["link_kind", "provenance"], as_index=False)
    .agg(entries=("rows", "size"), rows=("rows", "sum"))
    .sort_values(["link_kind", "provenance"])
)
summary

## Detail on the curated and mapping cases

These are the links that need a justification. Everything else is an authority
assertion carried by a release.

In [ ]:
instruments = sheet(INSTRUMENT_IDENTITY, "Instruments")
match_methods = instruments["Instrument_Match_Method"].value_counts(dropna=False).rename_axis("method").reset_index(name="codelists")
match_methods

In [ ]:
coding = sheet(GRAPH, "Coding")
external_systems = (
    coding.groupby(["system_name", "system"], as_index=False)
    .agg(rows=("code", "size"), distinct_codes=("code", "nunique"), distinct_bcs=("bc_id", "nunique"))
)
external_systems

### LOINC at two grains

BC-level (`Coding`) versus DSS-level (a `--LOINC` variable with a pinned
value). The question for RDF: the BC-level link is a mapping statement, the
DSS-level one is a value on a variable in a fixed context.

In [ ]:
variables = sheet(GRAPH, "Variables")
bc_loinc = coding[coding["system_name"] == "LOINC"][["bc_id", "code"]].rename(columns={"code": "bc_level_loinc"})
loinc_vars = variables[variables["variable_name"].str.endswith("LOINC", na=False)]
assigned_loinc = loinc_vars[loinc_vars["assigned_term_value"].notna()][
    ["ds_id", "bc_id", "assigned_term_value"]
].rename(columns={"assigned_term_value": "dss_level_loinc"})
listed_loinc = loinc_vars[loinc_vars["value_list"].notna()][["ds_id", "bc_id", "value_list"]].copy()
listed_loinc["dss_level_loinc"] = listed_loinc["value_list"].str.split(";")
listed_loinc = listed_loinc.explode("dss_level_loinc")[["ds_id", "bc_id", "dss_level_loinc"]]
listed_loinc["dss_level_loinc"] = listed_loinc["dss_level_loinc"].str.strip()
dss_loinc = pd.concat([assigned_loinc, listed_loinc], ignore_index=True)

both = dss_loinc.merge(bc_loinc, on="bc_id", how="left")
fanout = dss_loinc.groupby("bc_id")["dss_level_loinc"].nunique()

grain = pd.DataFrame(
    [
        {"measure": "BCs with a BC-level LOINC", "count": bc_loinc["bc_id"].nunique()},
        {"measure": "DSSs pinning a LOINC value", "count": dss_loinc["ds_id"].nunique()},
        {"measure": "--LOINC variables", "count": len(loinc_vars)},
        {"measure": "... with one assigned value", "count": assigned_loinc["ds_id"].nunique()},
        {"measure": "... with a value list", "count": listed_loinc["ds_id"].nunique()},
        {"measure": "... with no value", "count": int((loinc_vars["assigned_term_value"].isna() & loinc_vars["value_list"].isna()).sum())},
        {"measure": "DSS x LOINC pairs", "count": len(dss_loinc)},
        {"measure": "distinct LOINC codes at DSS level", "count": dss_loinc["dss_level_loinc"].nunique()},
        {"measure": "pinned DSSs whose BC also has a BC-level LOINC", "count": both.loc[both["bc_level_loinc"].notna(), "ds_id"].nunique()},
        {"measure": "of those, BC-level code is among the DSS codes", "count": both.loc[both["bc_level_loinc"] == both["dss_level_loinc"], "ds_id"].nunique()},
        {"measure": "BCs whose DSSs pin more than one distinct LOINC", "count": int((fanout > 1).sum())},
    ]
)
grain

### Value lists

`value_list` is a `;`-separated string on the variable. The audit table above
counts variables; this counts the values behind them.

In [ ]:
listed = variables[variables["value_list"].notna()].copy()
both_set = listed["assigned_term_concept_id"].notna() | listed["assigned_term_value"].notna()
if both_set.any():
    raise ValueError(f"{int(both_set.sum())} variables carry both a value list and an assigned value")

listed["scope"] = "no codelist"
listed.loc[listed["codelist_concept_id"].notna(), "scope"] = "bound codelist"
listed.loc[listed["variable_name"].str.endswith("LOINC"), "scope"] = "LOINC"
listed["value"] = listed["value_list"].str.split(";")
exploded = listed.explode("value")
exploded["value"] = exploded["value"].str.strip()

value_list_detail = pd.DataFrame({
    "variables": listed.groupby("scope").size(),
    "dss": listed.groupby("scope")["ds_id"].nunique(),
    "with_subset_codelist": listed.groupby("scope")["subset_codelist"].count(),
    "values_total": exploded.groupby("scope").size(),
    "distinct_values": exploded.groupby("scope")["value"].nunique(),
}).reset_index()
value_list_detail

### Assignment states of CT-bound variables

For every variable with a bound codelist, one of three states: `assigned` (one
value fixed), `value_list` (restricted to a list), `open` (the whole codelist).
Grouped by variable suffix (variable name minus the domain prefix). Measured
only - no reading is attached to the states here.

In [ ]:
dss = sheet(GRAPH, "DSS")
ct_bound = variables[variables["codelist_concept_id"].notna()].merge(dss[["ds_id", "domain"]], on="ds_id")
ct_bound["suffix"] = [
    "--" + name[len(domain):] if name.startswith(domain) else name
    for name, domain in zip(ct_bound["variable_name"], ct_bound["domain"])
]
ct_bound["state"] = "open"
ct_bound.loc[ct_bound["value_list"].notna(), "state"] = "value_list"
ct_bound.loc[
    ct_bound["assigned_term_concept_id"].notna() | ct_bound["assigned_term_value"].notna(), "state"
] = "assigned"

states = pd.crosstab(ct_bound["suffix"], ct_bound["state"]).reindex(
    columns=["assigned", "value_list", "open"], fill_value=0
)
states.insert(0, "ct_bound", states.sum(axis=1))
states["assigned_rate"] = (states["assigned"] / states["ct_bound"]).round(2)
states = states.sort_values(["ct_bound"], ascending=False).reset_index()
print(len(ct_bound), "CT-bound variables,", len(states), "suffixes")
states.head(30)

### Category tokens

Same mechanism as `cosmos-graph/notebooks/50_instrument_category_resolution.ipynb`
(which stays the authority for it), applied to all tokens instead of the
instrument scope: exact `bc_short_name`, then a unique `bc_synonyms` match
(lower-cased, `;`-split). Deterministic only.

In [ ]:
bc = sheet(GRAPH, "BC")
bc_categories = sheet(GRAPH, "BC_Categories")

shortname_to_bc = {}
for bc_id, short_name in zip(bc["bc_id"], bc["bc_short_name"]):
    shortname_to_bc.setdefault(str(short_name).strip(), []).append(bc_id)

synonym_to_bc = {}
for bc_id, synonyms in zip(bc["bc_id"], bc["bc_synonyms"]):
    if pd.isna(synonyms):
        continue
    for synonym in str(synonyms).split(";"):
        synonym = synonym.strip()
        if synonym:
            synonym_to_bc.setdefault(synonym.lower(), set()).add(bc_id)


def resolve(token):
    if token in shortname_to_bc:
        return "exact_name", shortname_to_bc[token]
    hits = synonym_to_bc.get(token.lower())
    if hits:
        if len(hits) == 1:
            return "synonym", sorted(hits)
        return "synonym_ambiguous", sorted(hits)
    return "unresolved", []


uses = bc_categories["category"].value_counts()
records = []
for token in sorted(uses.index):
    status, bc_ids = resolve(token)
    records.append({
        "category": token,
        "status": status,
        "bc_id": ";".join(map(str, bc_ids)),
        "uses": int(uses[token]),
    })

category_resolution = pd.DataFrame(records)
category_status = (
    category_resolution.groupby("status", as_index=False)
    .agg(tokens=("category", "size"), uses=("uses", "sum"))
)

raw_uses = int(bc["bc_categories"].dropna().str.split(";").str.len().sum())
print(raw_uses, "raw token uses in BC.bc_categories,", len(bc_categories), "distinct (BC, token) rows")
category_status

In [ ]:
category_resolution[category_resolution["status"] == "unresolved"].sort_values("uses", ascending=False).head(15)

### Integrity checks already in the graph

`Unresolved` = concept IDs not found in the CT package. `Anomalies` =
pinned term not in the bound codelist. Both are membership checks, i.e.
validation of published content against published content.

In [ ]:
checks = pd.DataFrame(
    [
        {"check": "Unresolved concept IDs", "rows": len(sheet(GRAPH_CT, "Unresolved"))},
        {"check": "Anomalies (pinned term not in bound codelist)", "rows": len(sheet(GRAPH_CT, "Anomalies"))},
    ]
)
checks

## Write the audit workbook

In [ ]:
with pd.ExcelWriter(OUT, engine="openpyxl") as writer:
    audit.to_excel(writer, sheet_name="Link_Kinds", index=False)
    summary.to_excel(writer, sheet_name="Summary", index=False)
    match_methods.to_excel(writer, sheet_name="Instrument_Match_Method", index=False)
    external_systems.to_excel(writer, sheet_name="External_Systems", index=False)
    grain.to_excel(writer, sheet_name="LOINC_Grains", index=False)
    pins.to_excel(writer, sheet_name="Pins", index=False)
    value_list_detail.to_excel(writer, sheet_name="Value_List_Detail", index=False)
    states.to_excel(writer, sheet_name="Assignment_States", index=False)
    category_status.to_excel(writer, sheet_name="Category_Status", index=False)
    category_resolution.to_excel(writer, sheet_name="Category_Resolution", index=False)
    checks.to_excel(writer, sheet_name="Integrity_Checks", index=False)

print("written:", OUT)

## How to read this

- `identity`, `membership` and `binding` rows are **not** mappings. In RDF they
  need a predicate that is not transitive, so that merging graphs cannot create
  equivalences that no source asserts.
- `denotation` rows link a model element to a concept. `skos:exactMatch` treats
  the two as peers and is transitive; an annotation property does not.
- `pinned_value` rows are values in a fixed context, not statements about
  concept equivalence. The context is the DSS.
- `value_list` rows restrict a variable to a set of values in a fixed context.
  They sit between `binding` (the whole codelist) and `pinned_value` (one
  value). A value list and an assigned value never occur on the same variable.
- `category_tag` rows have a label at the object end, not an identifier. The
  same holds for the `subset_codelist` names and for value lists with no bound
  codelist. The category resolution status says which tokens can be tied to a
  BC by a deterministic join; it is derived, not published.
- `mapping` rows are the only ones where a justification is meaningful. Two of
  them already carry one: `Instrument_Match_Method` and the cross-reference
  `rationale`.

Next: the code-collision check - every C-code used in more than one role, which
is where a transitive predicate would bite in a merged graph.